## Top 20 priority counties

The headline actionable deliverable. A ranked list of MS counties most in need of state intervention, with a recommended action type per county. This is what gets shown to MSDH leadership.

### Importing necessary libraries

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine, text
import os
from dotenv import load_dotenv

load_dotenv('../.env')
host = os.getenv('PGHOST', 'localhost')
port = os.getenv('PGPORT', '5432')
db = os.getenv('PGDATABASE', 'ms_health')
user = os.getenv('PGUSER', 'postgres')
pwd = os.getenv('PGPASSWORD', '')
engine = create_engine(f'postgresql+psycopg://{user}:{pwd}@{host}:{port}/{db}')
sns.set(style='whitegrid')

def q(sql):
    return pd.read_sql(text(sql), engine)

### The actionable top 20 list

In [2]:
# Top 20 priority counties joined with two pieces of context per county:
# count of birthing friendly hospitals and Primary Care HPSA score.
# A CASE expression assigns each county a recommended intervention type
# from the four operational categories MSDH funds: mobile OB, coverage
# outreach, community health workers, and chronic disease management.
top20 = q("""
SELECT
    p.priority_rank AS rank,
    p.county_name,
    p.region,
    CASE WHEN p.is_delta THEN 'Delta' ELSE '' END AS delta,
    p.county_pop AS population,
    p.priority_score AS priority,
    p.county_mri AS mri,
    p.pct_in_care_desert || '%' AS pct_in_care_desert,
    COALESCE(p.imr_per_1000::TEXT, 'suppressed') AS imr,
    p.svi_overall_pct AS svi_pct,
    p.pct_uninsured || '%' AS uninsured,
    (SELECT COUNT(*) FROM dim_facility f
        WHERE f.county_fips = p.county_fips AND f.is_birthing_friendly) AS birthing_hospitals,
    (SELECT MAX(avg_hpsa_score) FROM fact_hpsa_county h
        WHERE h.county_fips = p.county_fips AND h.discipline = 'Primary Care') AS primary_care_hpsa_score,
    CASE
        WHEN p.pct_in_care_desert > 50 AND p.county_mri > 70 THEN 'STAND UP MOBILE OB CARE'
        WHEN p.county_mri > 70 AND p.pct_uninsured > 20 THEN 'EXPAND COVERAGE OUTREACH'
        WHEN p.svi_overall_pct > 0.85 AND p.county_mri > 60 THEN 'COMMUNITY HEALTH WORKER PROGRAM'
        ELSE 'TARGETED CHRONIC DISEASE INTERVENTION'
    END AS recommended_intervention
FROM mart_top20_priority p
WHERE p.priority_rank <= 20
ORDER BY p.priority_rank
""")
top20

,rank,county_name,region,delta,population,priority,mri,pct_in_care_desert,imr,svi_pct,uninsured,birthing_hospitals,primary_care_hpsa_score,recommended_intervention
0,1,Holmes,Delta,Delta,16848,81.6,62.6,100.0%,14.6,0.963,12.1%,0,23.000000,COMMUNITY HEALTH WORKER PROGRAM
1,2,Sharkey,Delta,Delta,3910,77.7,63.5,0.0%,17.6,0.938,20.3%,0,22.000000,COMMUNITY HEALTH WORKER PROGRAM
2,3,Claiborne,Other,,9044,77.0,55.5,57.4%,15.4,0.728,19.5%,0,19.500000,TARGETED CHRONIC DISEASE INTERVENTION
3,4,Yazoo,Delta,Delta,27467,76.9,58.4,88.4%,12.1,0.975,13.7%,0,20.500000,TARGETED CHRONIC DISEASE INTERVENTION
4,5,Tunica,Delta,Delta,9738,76.2,62.5,82.5%,13.1,0.877,13.1%,0,22.000000,COMMUNITY HEALTH WORKER PROGRAM
5,6,Leake,Other,,21335,75.4,48.9,100.0%,13.5,0.778,15.9%,0,22.000000,TARGETED CHRONIC DISEASE INTERVENTION
6,7,Washington,Delta,Delta,44604,62.9,55.7,0.0%,16.7,0.988,13.4%,1,23.000000,TARGETED CHRONIC DISEASE INTERVENTION
7,8,Attala,Other,,17842,59.9,45.5,100.0%,11.1,0.691,9.9%,0,20.000000,TARGETED CHRONIC DISEASE INTERVENTION
8,9,Coahoma,Delta,Delta,21264,59.3,58.4,0.0%,13,0.951,13.4%,1,23.500000,TARGETED CHRONIC DISEASE INTERVENTION
9,10,Bolivar,Delta,Delta,30688,59.0,55.1,0.0%,13,0.926,14.6%,1,23.666667,TARGETED CHRONIC DISEASE INTERVENTION


### Triple burden tracts inside those counties

In [3]:
# Triple burden tracts: top-quintile MRI, more than 30 miles from nearest birthing
# hospital, and SVI overall percentile at or above 0.8. These are the specific
# tract-level addresses for MSDH to target first.
triple = q("""
WITH base AS (
    SELECT m.tract_fips, g.county_name, g.region, g.is_delta, g.total_population,
        m.mri, m.mri_quintile,
        m.pre_existing_avg, m.mental_health_avg,
        d.distance_miles_rounded AS dist_to_birthing,
        d.est_drive_minutes,
        d.nearest_hospital_name,
        s.rpl_themes AS svi,
        s.ep_uninsur AS pct_uninsured,
        s.ep_noveh AS pct_no_vehicle,
        s.ep_noint AS pct_no_internet
    FROM mart_maternal_risk_index m
    JOIN dim_geography g ON g.tract_fips = m.tract_fips
    LEFT JOIN mart_drive_time d ON d.tract_fips = m.tract_fips
    LEFT JOIN fact_svi_wide s ON s.geo_sk = g.geo_sk
)
SELECT tract_fips, county_name, region, mri,
    dist_to_birthing AS miles_to_l_and_d,
    est_drive_minutes AS drive_minutes,
    nearest_hospital_name,
    ROUND(svi::NUMERIC, 3) AS svi_overall,
    pct_uninsured, pct_no_vehicle, pct_no_internet
FROM base
WHERE mri_quintile = 5
AND dist_to_birthing > 30
AND svi >= 0.8
ORDER BY mri DESC
""")
triple

,tract_fips,county_name,region,mri,miles_to_l_and_d,drive_minutes,nearest_hospital_name,svi_overall,pct_uninsured,pct_no_vehicle,pct_no_internet
0,28163950500,Yazoo,Delta,79.9,39.7,53.0,UNIVERSITY OF MISSISSIPPI MED CENTER,0.960,18.2,23.5,26.8
1,28163950300,Yazoo,Delta,76.3,41.8,55.7,UNIVERSITY OF MISSISSIPPI MED CENTER,0.982,14.8,20.9,22.3
2,28143950102,Tunica,Delta,71.0,41.0,54.6,NORTHWEST MISSISSISSIPPI REGIONAL MEDICAL CENTER,0.879,17.6,5.5,5.5
3,28051950501,Holmes,Delta,69.1,46.3,61.8,MERIT HEALTH WOMEN'S HOSPITAL,0.956,12.3,21.7,40.1
4,28099940100,Neshoba,Other,67.6,43.3,57.8,OCHSNER RUSH HOSPITAL,0.966,28.9,16.3,25.3
5,28051950300,Holmes,Delta,65.0,32.1,42.8,SOUTH SUNFLOWER COUNTY HOSPITAL,0.893,13.6,15.6,45.7
6,28093950201,Marshall,Other,65.0,37.0,49.3,BAPTIST MEMORIAL HOSPITAL NORTH MS,0.912,19.3,2.3,26.0
7,28143950200,Tunica,Delta,64.1,33.6,44.8,NORTHWEST MISSISSISSIPPI REGIONAL MEDICAL CENTER,0.894,11.1,16.3,24.2
8,28051950100,Holmes,Delta,63.4,44.4,59.2,UNIVERSITY OF MISSISSIPPI MEDICAL CENTER- GRENADA,0.941,15.8,12.3,33.0
9,28051950502,Holmes,Delta,63.3,51.2,68.3,MERIT HEALTH WOMEN'S HOSPITAL,0.852,8.8,24.6,30.2


### Counties with zero birthing friendly hospitals

In [4]:
# Counties with zero birthing-friendly hospitals, plus the count of women of
# reproductive age affected per county.
coverage = q("""
WITH counties AS (
    SELECT DISTINCT county_fips, county_name, region, is_delta
    FROM dim_geography
),
hospitals_per_county AS (
    SELECT county_fips,
        COUNT(*) FILTER (WHERE is_birthing_friendly) AS n_birthing_friendly,
        STRING_AGG(facility_name, '; ' ORDER BY facility_name) AS facility_list
    FROM dim_facility
    WHERE state_abbr = 'MS'
    GROUP BY county_fips
),
women_per_county AS (
    SELECT g.county_fips, SUM(a.women_reproductive_age) AS women_15_44
    FROM dim_geography g
    LEFT JOIN acs_headline a ON a.tract_fips = g.tract_fips
    GROUP BY g.county_fips
)
SELECT c.county_name, c.region,
    COALESCE(h.n_birthing_friendly, 0) AS birthing_friendly_hospitals,
    COALESCE(w.women_15_44, 0) AS women_15_44
FROM counties c
LEFT JOIN hospitals_per_county h USING (county_fips)
LEFT JOIN women_per_county w USING (county_fips)
WHERE COALESCE(h.n_birthing_friendly, 0) = 0
ORDER BY women_15_44 DESC
""")

print(f'Counties with no birthing-friendly hospital: {len(coverage)}')
print(f'Women aged 15 to 44 in those counties: {coverage.women_15_44.sum():,.0f}')
coverage.head(20)

Counties with no birthing-friendly hospital: 54
Women aged 15 to 44 in those counties: 111,516


,county_name,region,birthing_friendly_hospitals,women_15_44
0,DeSoto,Delta,0,18557.0
1,Madison,Jackson Metro,0,10555.0
2,Hancock,Gulf Coast,0,3600.0
3,Marshall,Other,0,3313.0
4,Panola,Delta,0,3278.0
5,Tate,Delta,0,3029.0
6,Pontotoc,Other,0,2964.0
7,Leflore,Delta,0,2870.0
8,Neshoba,Other,0,2856.0
9,Copiah,Other,0,2849.0
